# 02 – ViLT (LoRA) · eğitim ve güvenilirlik denetimi

Eğitim → en iyi (doğrulama macro-AUC) kontrol noktası → **sabit test ızgarasında** orig/noise/black/shuffle
→ istatistikler → (isteğe bağlı) IoU. Mantığın tamamı `src/` altındadır; bu notebook yalnızca akışı çalıştırır.

> **ViLT notu:** Eski kodda sınıflandırma başlığı PEFT sarmalının içinde kaldığı için (donuk, rastgele başlangıç) eğitilmiyor olabilirdi. Burada başlık LoRA dışındadır ve eğitilir (`param_report` ile doğrulayın).

**Önce `SMOKE = True` ile çalıştırın** (≈ dakikalar): uçtan uca hata var mı görürsünüz. Sonra `SMOKE = False`.

In [ ]:
MODEL   = "vilt"        # "blip" | "vilt" | "clip"
SMOKE   = False            # True: küçük duman testi (kontrol noktaları/sonuçlar *_smoke klasörlerine gider)
RETRAIN = False            # True: mevcut kontrol noktası olsa da yeniden eğit
RERUN_INFERENCE = False    # True: kaydedilmiş test tahminleri olsa da yeniden hesapla
RUN_IOU = True            # sayısal IoU (VinDr DICOM + annotations_train.csv gerekir)
RUN_CAM = True             # örnek Grad-CAM / dikkat haritası görseli (VinDr annotations gerekir)
EPOCHS  = 10               # tam eğitimde epoch sayısı (süreyi kısaltmak için 5)

SPLIT_DIR = "splits"
CKPT_DIR  = "checkpoints_smoke" if SMOKE else "checkpoints"
RES_DIR   = "results_smoke"     if SMOKE else "results"
N_BOOT    = 20 if SMOKE else 500

VINDR_ANN = "vindr/annotations_train.csv"
DICOM_DIR = "path/to/vindr-cxr/1.0.0/train/"   # kendi yolunuz

In [ ]:
import sys, os, json; sys.path.insert(0, "src")
import numpy as np, pandas as pd, torch
import cxr_data as D, cxr_models as M, cxr_eval as E
import transformers, peft

M.seed_everything(); device = M.get_device()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("device:", device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
for d in (CKPT_DIR, RES_DIR): os.makedirs(d, exist_ok=True)

sp = D.load_splits(SPLIT_DIR)              # sızıntı kontrolü load içinde yapılır (assert)
display(D.leakage_report(sp))

## 1. Eğitim

In [ ]:
SM = dict(max_train_images=2000, max_val_images=128, epochs=1, num_workers=2) if SMOKE else dict(epochs=EPOCHS)
CKPT_BEST = f"{CKPT_DIR}/{MODEL}_best.pt"
if os.path.exists(CKPT_BEST) and not RETRAIN:
    print("Kontrol noktası mevcut, eğitim atlandı:", CKPT_BEST)
else:
    log = M.train_model(MODEL, sp, out_dir=CKPT_DIR, device=device, **SM)
    display(log)

In [ ]:
import matplotlib.pyplot as plt
lg = pd.read_csv(f"{CKPT_DIR}/{MODEL}_train_log.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(lg.epoch, lg.train_loss, label="train"); ax[0].plot(lg.epoch, lg.val_loss, label="val"); ax[0].legend(); ax[0].set_title("loss")
ax[1].plot(lg.epoch, lg.macro_auc, label="macro"); ax[1].plot(lg.epoch, lg.micro_auc, label="micro"); ax[1].legend(); ax[1].set_title("doğrulama AUC")
plt.show()

## 2. En iyi kontrol noktasını yükle (strict=True) ve test ızgarasında çalıştır
`load_checkpoint` eksik/fazla anahtar olursa **hata verir** (eski `strict=False` sessizce atlıyordu).

In [ ]:
model, proc = M.build_model(MODEL, device)
enc = M.Encoder(MODEL, proc)
ck = M.load_checkpoint(model, CKPT_BEST, device)
print("seçilen epoch:", ck.get("epoch"), "| doğrulama metrikleri:", {k: round(v, 4) for k, v in ck.get("metrics", {}).items() if isinstance(v, float)})
display(M.param_report(model))

test_df = sp.test.sample(200, random_state=0) if SMOKE else sp.test
grid = D.build_eval_grid(test_df, sp.label_cols)
print("test çifti:", len(grid), "| görüntü:", grid.img_idx.nunique())

In [ ]:
pred_path = f"{RES_DIR}/{MODEL}_test_predictions.csv.gz"
if os.path.exists(pred_path) and not RERUN_INFERENCE:
    pred = pd.read_csv(pred_path)
else:
    pred = E.run_modes(model, grid, enc, device, num_workers=2 if SMOKE else 4)
    pred.to_csv(pred_path, index=False)
pred["prob_prior"] = E.text_prior_baseline(pred, sp.train, sp.label_cols)   # görüntüye bakmayan taban çizgisi
print(pred.shape); pred.head()

## 3. Ana tablo (Tablo 3 / Tablo 7 yerine)
* **Full**: orijinal görüntü · **Noise**: görüntü yerine N(0,1) gürültü · **Black**: sabit gerçek siyah görüntü
* **Text prior**: yalnızca *hangi hastalığın sorulduğu* (eğitim prevalansı) — görüntüsüz taban çizgisi
* **Shuffle-k**: k×k piksel yamaları, görüntü başına bağımsız permütasyon

In [ ]:
cols = {"Full": "prob_orig", "Noise": "prob_noise", "Black": "prob_black", "Text prior": "prob_prior",
        "Shuffle-32": "prob_shuf32", "Shuffle-16": "prob_shuf16", "Shuffle-8": "prob_shuf8"}
H = E.headline_table(pred, cols)
H.to_csv(f"{RES_DIR}/{MODEL}_headline.csv", index=False); display(H.round(4))

In [ ]:
pdz = E.per_disease_table(pred, "prob_orig", min_pos=30, min_neg=30)
pdz.to_csv(f"{RES_DIR}/{MODEL}_per_disease_full.csv", index=False)
display(pdz.round(4))
print("Güvenilmez (n_pos<30 veya n_neg<30) hastalıklar:", pdz.loc[~pdz.reliable, "disease"].tolist())

## 4. Görsel bağımlılık — geçerli analiz
Sabit girdide (Black) çıktı yalnızca soruya bağlıdır → **hastalık-bazlı AUC her zaman 0,5** (aşağıdaki kontrol).
Bu nedenle eski "VRI = AUC_full − AUC_blind (hastalık bazlı)" tanımsızdır. Geçerli göstergeler:
eşleşik ΔP (Wilcoxon, Cohen dz), micro AUC (Full vs Black vs Text-prior).

In [ ]:
chk = E.per_disease_table(pred, "prob_black", min_pos=1, min_neg=1)
print("Black altında hastalık-bazlı AUC aralığı:", chk.auc.min().round(4), "-", chk.auc.max().round(4), "(≈0.5 beklenir)")
vd = E.visual_dependence_table(pred); vd.to_csv(f"{RES_DIR}/{MODEL}_visual_dependence.csv", index=False)
display(vd.round(4))

## 5. Güven aralıkları (hasta-küme bootstrap)

In [ ]:
bs = E.bootstrap_auc(pred, ["prob_orig", "prob_black", "prob_prior", "prob_shuf32", "prob_shuf16", "prob_shuf8"],
                     diffs=[("prob_orig", "prob_shuf32"), ("prob_orig", "prob_shuf16"), ("prob_orig", "prob_shuf8"),
                            ("prob_orig", "prob_black"), ("prob_orig", "prob_prior")],
                     n_boot=N_BOOT)
bs.to_csv(f"{RES_DIR}/{MODEL}_bootstrap_micro_auc.csv", index=False); display(bs.round(4))

## 6. (İsteğe bağlı) Grad-CAM / dikkat ve IoU
Model, eğitimdeki sınıfla + LoRA ile + `strict=True` yüklendi (eski IoU hücreleri LoRA ağırlıklarını düşürüyordu).
Yalnızca **test bölmesindeki** VinDr görüntüleri; en iyi-eşleşme IoU, pointing-hit ve **sabit merkez kutusu taban çizgisi** raporlanır.

In [ ]:
if RUN_IOU:
    iou_det, iou_sum, skipped = E.run_iou(MODEL, model, enc, sp.test, VINDR_ANN, DICOM_DIR, device,
                                          max_images_per_disease=10 if SMOKE else None)
    iou_det.to_csv(f"{RES_DIR}/{MODEL}_iou_detail.csv", index=False)
    iou_sum.to_csv(f"{RES_DIR}/{MODEL}_iou_summary.csv", index=False)
    print("atlananlar:", skipped); display(iou_sum.round(4))
else:
    print("RUN_IOU=False; atlandı.")

In [ ]:
summary = dict(model=MODEL, smoke=SMOKE, best_epoch=ck.get("epoch"), n_test_pairs=int(len(pred)),
               n_test_images=int(pred.img_idx.nunique()),
               torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,
               headline=H.round(4).to_dict(orient="records"))
json.dump(summary, open(f"{RES_DIR}/{MODEL}_run_summary.json", "w"), indent=1, ensure_ascii=False)
print("kaydedildi:", f"{RES_DIR}/{MODEL}_run_summary.json")

## 7. Örnek ısı haritaları (görsel kontrol)
BLIP/CLIP: Grad-CAM · ViLT: son katman dikkat haritası (metin [CLS] -> görüntü yamaları).
**Yeşil** = radyolog kutusu, **kırmızı kesikli** = modelin tahmin kutusu (eşik 0.5).
Sonuçlar diske kaydedildikten sonra çalışır; hata olsa bile yukarıdaki çıktılar korunur.

In [ ]:
if RUN_CAM:
    import matplotlib.pyplot as plt
    ann = pd.read_csv(VINDR_ANN).dropna(subset=["x_min", "y_min", "x_max", "y_max"])
    vt = sp.test[sp.test.source == "vindr"].copy()
    vt["image_id"] = vt.image_path.map(lambda p: os.path.splitext(os.path.basename(p))[0])
    id2path = dict(zip(vt.image_id, vt.image_path))
    items = []
    for dis in ["Aortic enlargement", "Cardiomegaly", "Pleural thickening"]:
        a = ann[(ann.class_name == dis) & ann.image_id.isin(set(id2path))]
        if a.empty:
            continue
        img_id = a.image_id.iloc[0]
        Wd, Hd, _ = E._image_size(img_id, id2path[img_id], DICOM_DIR)     # (H adı kullanılmaz: ana tabloyu ezmesin)
        gts = [[r.x_min / Wd, r.y_min / Hd, r.x_max / Wd, r.y_max / Hd] for r in a[a.image_id == img_id].itertuples()]
        items.append((id2path[img_id], dis, gts))
    fig = E.plot_cam_examples(model, MODEL, enc, items, device, path=f"{RES_DIR}/{MODEL}_cam_examples.png")
    plt.show()